# W2D1 — Blur 비교: GaussianBlur vs medianBlur vs bilateralFilter

## 📌 오늘의 목표
3가지 blur 함수의 **특성 차이**를 직접 눈으로 확인하고,  
어떤 결함 검사 상황에서 어떤 blur를 선택해야 하는지 판단 기준을 세운다.

## 🎯 배울 함수
| 함수 | 핵심 파라미터 | 특징 |
|------|-------------|------|
| `cv2.GaussianBlur` | `ksize`, `sigmaX` | 가우시안 분포 가중치, 부드러운 흐림 |
| `cv2.medianBlur` | `ksize` | 중간값 사용, salt-and-pepper 노이즈에 강함 |
| `cv2.bilateralFilter` | `d`, `sigmaColor`, `sigmaSpace` | 엣지 보존하면서 내부만 흐림 |

## 📖 원문 자료
> 📖 원리 이해: [LearnOpenCV - Image Filtering](https://learnopencv.com/image-filtering-using-convolution-in-opencv/)  
> 📖 원리 이해: [LearnOpenCV - Bilateral Filtering](https://learnopencv.com/non-photorealistic-rendering-using-weaving-and-hatching/)  
> 📋 파라미터 확인: [OpenCV 공식 - Smoothing](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html)

## 🏭 실무 연결
AOI/SPI 검사에서 blur는 "전처리의 첫 단계"다.  
카메라 센서 노이즈, 조명 불균일, 표면 질감 같은 **배경 잡음을 제거**하면서  
결함 신호(엣지, 밝기 차이)는 살려야 한다 → **어떤 blur를 쓰느냐가 검출률을 결정**한다.

---
## 📦 1. Import & 데이터 로딩

**왜 grayscale로 로드하나?**  
→ 제조 결함 검사는 색상보다 **밝기 차이**로 결함을 판단한다.  
→ 흑백 이미지는 채널이 1개라 처리 속도가 3배 빠르고, blur 효과가 눈에 더 잘 보인다.

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_ROOT = Path('../data/kaggle/archive/NEU-DET/train/images')

img_crazing = cv2.imread(str(DATA_ROOT / 'crazing/crazing_1.jpg'), cv2.IMREAD_GRAYSCALE)
img_scratches = cv2.imread(str(DATA_ROOT / 'scratches/scratches_1.jpg'), cv2.IMREAD_GRAYSCALE)
img_pitted = cv2.imread(str(DATA_ROOT / 'pitted_surface/pitted_surface_1.jpg'), cv2.IMREAD_GRAYSCALE)

print(f'이미지 크기: {img_crazing.shape}')
print(f'픽셀 값 범위: {img_crazing.min()} ~ {img_crazing.max()}')

---
## 🔧 2. 세 가지 Blur 비교 — 같은 커널 사이즈로

### cv2.GaussianBlur
- **원리**: 커널 중심에 가까울수록 높은 가중치로 픽셀 평균 → 부드럽게 퍼짐
- **파라미터**: `ksize=(15, 15)` — 영향 받는 픽셀 범위 (반드시 홀수), `sigmaX=0` — 자동 계산
- **실무 용도**: 전반적인 배경 노이즈 제거, 가장 범용적으로 쓰임

### cv2.medianBlur
- **원리**: 커널 내 픽셀들의 중간값으로 대체 → 튀는 값(노이즈)이 선택될 확률 0
- **파라미터**: `ksize=15` — 홀수만 가능
- **실무 용도**: 센서 먼지, 전기 노이즈로 인한 점 노이즈(salt-and-pepper) 제거

### cv2.bilateralFilter
- **원리**: 공간 거리 + **픽셀 값 차이**를 함께 가중치로 사용 → 비슷한 밝기끼리만 혼합
- **파라미터**: `d=15` — 영향 반경, `sigmaColor=75` — 색상 허용 범위, `sigmaSpace=75` — 공간 범위
- **실무 용도**: 결함 경계(엣지)를 살리면서 표면 질감 노이즈만 제거

In [ ]:
img = img_crazing

blur_gaussian = cv2.GaussianBlur(img, ksize=(15, 15), sigmaX=0)
blur_median = cv2.medianBlur(img, ksize=15)
blur_bilateral = cv2.bilateralFilter(img, d=15, sigmaColor=75, sigmaSpace=75)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
titles = ['원본', 'GaussianBlur', 'medianBlur', 'bilateralFilter']
images = [img, blur_gaussian, blur_median, blur_bilateral]

for ax, title, image in zip(axes, titles, images):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.suptitle('Crazing — Blur 3종 비교 (ksize=15)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
1. **결함 선(crazing 균열)** — 세 blur 후에 선이 얼마나 흐려졌는지 비교
2. **배경 영역** — 얼마나 균일하게 정리됐는지
3. **bilateralFilter** — 결함 경계가 다른 두 방법보다 선명하게 남아 있는지

> 🤔 **판단 질문 1:** bilateralFilter 결과에서 균열의 경계가 GaussianBlur보다 선명하게 보이나요? 보인다면 왜 그런 걸까요?  
> 🤔 **판단 질문 2:** medianBlur와 GaussianBlur의 배경 노이즈 제거 정도가 눈에 띄게 다른가요, 비슷한가요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 1 [예측형]: ksize를 15에서 3으로 줄이면 세 blur 결과가 어떻게 변할까? 예상 후 아래 셀에서 실행.
- 챌린지 2 [판단형]: 지금 bilateralFilter가 세 방법 중 가장 느린 이유는 뭘까? (힌트: 계산에 쓰이는 정보 개수)
- 챌린지 3 [지시형]: "crazing 대신 pitted_surface 이미지로 같은 비교를 해줘"라고 Claude에게 지시해보세요.
```

#### ✏️ 빈칸 채우기 — 챌린지 1: ksize=3으로 blur 3종 재실행
> `___` 부분을 채워서 실행하세요. 예측과 결과가 같나요?

In [ ]:
# ksize=3으로 변경 — ___ 부분을 채우세요
blur_gaussian_s  = cv2.GaussianBlur(img, ksize=(___,___), sigmaX=___)
blur_median_s    = cv2.medianBlur(img, ksize=___)
blur_bilateral_s = cv2.bilateralFilter(img, d=___, sigmaColor=75, sigmaSpace=75)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, title, image in zip(axes,
                             ['원본', 'Gaussian(k=3)', 'Median(k=3)', 'Bilateral(d=3)'],
                             [img, blur_gaussian_s, blur_median_s, blur_bilateral_s]):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')
plt.suptitle('ksize=3 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 3. 차이 이미지 — blur가 뭘 지웠는지 확인

**왜 차이 이미지를 보나?**  
→ `원본 - blur 결과 = blur가 제거한 성분`이다.  
→ 이 차이가 **노이즈인지 결함 신호인지** 확인해야 blur 선택이 맞는지 판단할 수 있다.  
→ AOI에서는 "지워진 것이 결함 신호면 안 된다"는 기준으로 전처리를 검증한다.

**cv2.absdiff**: 두 이미지의 절대 차이 계산 (W1에서 배운 함수)  
→ `|원본 - blur|` = blur가 제거한 픽셀 정보량

In [ ]:
diff_gaussian = cv2.absdiff(img, blur_gaussian)
diff_median = cv2.absdiff(img, blur_median)
diff_bilateral = cv2.absdiff(img, blur_bilateral)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['원본 - GaussianBlur', '원본 - medianBlur', '원본 - bilateralFilter']
diffs = [diff_gaussian, diff_median, diff_bilateral]

for ax, title, diff in zip(axes, titles, diffs):
    ax.imshow(diff, cmap='hot')
    ax.set_title(f'{title}\n(평균 차이: {diff.mean():.2f})', fontsize=12)
    ax.axis('off')

plt.suptitle('Blur가 제거한 성분 (밝을수록 많이 지워짐)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'GaussianBlur  제거량: {diff_gaussian.mean():.2f}')
print(f'medianBlur    제거량: {diff_median.mean():.2f}')
print(f'bilateralFilter 제거량: {diff_bilateral.mean():.2f}')

### 👁 어디를 봐야 하나
1. **균열 선 부분(결함 영역)** — bilateralFilter의 차이 이미지에서 결함 부분이 어둡게(=적게 지워짐) 나타나는지
2. **배경 영역** — 세 방법 모두 배경에서 비슷한 양의 노이즈를 제거했는지
3. **평균 차이 수치** — 숫자가 클수록 더 강하게 blur됨

> 🤔 **판단 질문 3:** 차이 이미지에서 bilateralFilter의 결함 선 부분이 다른 두 방법보다 어둡다면, 이것이 "좋은 blur"의 증거인가요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 4 [예측형]: bilateralFilter의 sigmaColor를 75에서 10으로 낮추면 차이 이미지가 bilateralFilter에서 GaussianBlur에 가까워질까요, 멀어질까요?
- 챌린지 5 [판단형]: 평균 차이 수치가 가장 큰 방법이 "가장 좋은 blur"인가요? 판단 근거는?
```

#### ✏️ 빈칸 채우기 — 챌린지 4: sigmaColor를 낮추면 어떻게 되나?
> sigmaColor=10으로 줄이면 bilateral이 Gaussian에 가까워질까요, 멀어질까요? 예측 후 실행하세요.

In [ ]:
# sigmaColor=10 으로 변경 — ___ 부분을 채우세요
blur_bilateral_tight = cv2.bilateralFilter(img, d=15, sigmaColor=___, sigmaSpace=75)
diff_tight = cv2.absdiff(___, blur_bilateral_tight)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(diff_bilateral, cmap='hot')
axes[0].set_title(f'sigmaColor=75  평균: {diff_bilateral.mean():.2f}')
axes[0].axis('off')
axes[1].imshow(diff_tight, cmap='hot')
axes[1].set_title(f'sigmaColor=10  평균: {diff_tight.mean():.2f}')
axes[1].axis('off')
plt.suptitle('sigmaColor 변화 — bilateral 차이 이미지 비교', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 4. 파라미터 실험 — GaussianBlur ksize 변화

In [ ]:
ksizes = [3, 7, 15, 31]

fig, axes = plt.subplots(1, len(ksizes) + 1, figsize=(22, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('원본', fontsize=12)
axes[0].axis('off')

for ax, k in zip(axes[1:], ksizes):
    blurred = cv2.GaussianBlur(img, ksize=(k, k), sigmaX=0)
    ax.imshow(blurred, cmap='gray')
    ax.set_title(f'ksize={k}', fontsize=12)
    ax.axis('off')

plt.suptitle('GaussianBlur — ksize 파라미터 변화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- ksize가 커질수록 결함 선이 얼마나 사라지는지 추적
- **ksize=31** 수준에서는 결함 자체가 blur에 묻히지 않는지 확인

> 🤔 **판단 질문 4:** 제조 검사 전처리용 GaussianBlur라면 ksize를 얼마로 설정하는 게 적절할까요? 근거는?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 6 [예측형]: ksize를 짝수(예: 14)로 넣으면 어떻게 될까? 예상 후 실행.
- 챌린지 7 [지시형]: "medianBlur도 같은 ksize 변화 실험을 해줘"라고 Claude에게 지시해보세요.
```

#### ✏️ 빈칸 채우기 — 챌린지 6-7: 짝수 ksize & medianBlur ksize 실험
> **챌린지 6**: 짝수 ksize를 넣으면 어떤 에러 메시지가 나오나요?

> **챌린지 7**: GaussianBlur 실험과 같은 구조로 medianBlur 실험을 완성하세요.

In [ ]:
# 챌린지 6 — 짝수 ksize 실험 (에러 확인)
blurred_even = cv2.GaussianBlur(img, ksize=(___, ___), sigmaX=0)  # 짝수를 넣어보세요

# ---
# 챌린지 7 — medianBlur ksize 변화 실험 (GaussianBlur 셀과 같은 구조로 채우세요)
ksizes = [3, 7, 15, 31]
fig, axes = plt.subplots(1, len(ksizes) + 1, figsize=(22, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('원본', fontsize=12)
axes[0].axis('off')

for ax, k in zip(axes[1:], ksizes):
    blurred = cv2.___(img, ksize=___)  # 함수명과 파라미터를 채우세요
    ax.imshow(blurred, cmap='gray')
    ax.set_title(f'ksize={k}', fontsize=12)
    ax.axis('off')

plt.suptitle('medianBlur — ksize 파라미터 변화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 5. 결함 유형별 blur 비교 — scratches vs pitted_surface

**왜 결함 유형별로 비교하나?**  
→ blur 선택은 **결함의 구조**에 따라 달라진다.  
→ 선형 결함(scratches)은 bilateral이 선을 잘 보존하고,  
→ 점형 결함(pitted_surface)은 median이 노이즈 점과 결함 점을 구분하지 못할 수 있다.

In [ ]:
images_to_compare = {
    'Scratches': img_scratches,
    'Pitted Surface': img_pitted,
}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
col_titles = ['원본', 'GaussianBlur', 'medianBlur', 'bilateralFilter']

for row_idx, (defect_name, img_src) in enumerate(images_to_compare.items()):
    g = cv2.GaussianBlur(img_src, (15, 15), 0)
    m = cv2.medianBlur(img_src, 15)
    b = cv2.bilateralFilter(img_src, 15, 75, 75)

    for col_idx, image in enumerate([img_src, g, m, b]):
        axes[row_idx][col_idx].imshow(image, cmap='gray')
        axes[row_idx][col_idx].set_title(f'{defect_name}\n{col_titles[col_idx]}', fontsize=11)
        axes[row_idx][col_idx].axis('off')

plt.suptitle('결함 유형별 Blur 비교 (ksize=15)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- **Scratches 행**: bilateralFilter에서 긁힌 선(결함)의 경계가 살아 있는지
- **Pitted Surface 행**: 작은 구덩이(결함)가 medianBlur에서 사라지는지, 크기 임계값은?

> 🤔 **판단 질문 5:** Scratches에는 어떤 blur를 추천하고, Pitted Surface에는 어떤 blur를 추천하나요? 이유는?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 8 [판단형]: pitted_surface에서 medianBlur(ksize=15)가 구덩이를 메워버렸다면, 이건 전처리 성공인가 실패인가?
- 챌린지 9 [지시형]: "inclusion과 rolled-in_scale도 추가해서 6종 전체 비교 그리드를 만들어줘"라고 지시해보세요.
```

#### ✏️ 빈칸 채우기 — 챌린지 8-9: 결함 유형별 최적 blur 선택
> 각 결함에 가장 적합한 blur 함수와 파라미터를 직접 채워보세요.

```
선택지: cv2.GaussianBlur(img, (15,15), 0)
        cv2.medianBlur(img, 15)
        cv2.bilateralFilter(img, 15, 75, 75)
```

In [ ]:
# 각 결함 유형에 최적 blur를 선택해서 ___ 부분을 채우세요
# 힌트: 선형 결함은 엣지 보존, 점형 결함은 점 노이즈 제거가 핵심

best_for_scratches = cv2.___(img_scratches, ___)   # 긁힌 선 → 경계 보존 필요
best_for_pitted    = cv2.___(img_pitted, ___)       # 작은 구덩이 → 점 노이즈 제거

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, title, image in zip(axes,
                             ['Scratches 원본', 'Scratches 최적 blur',
                              'Pitted 원본', 'Pitted 최적 blur'],
                             [img_scratches, best_for_scratches,
                              img_pitted, best_for_pitted]):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle('결함 유형별 최적 blur 선택', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📝 오늘 배운 것 정리

### 핵심 의사결정 기준
| 상황 | 추천 blur | 이유 |
|------|----------|------|
| 전반적 노이즈 제거, 범용 | GaussianBlur | 빠르고 안정적 |
| 점 노이즈(센서 먼지, 전기 노이즈) | medianBlur | 중간값 특성상 튀는 값 제거 |
| 결함 엣지를 살려야 할 때 | bilateralFilter | 엣지 보존 특성, 단 느림 |

### 처리 속도 비교 (참고)
- GaussianBlur < medianBlur << bilateralFilter (느린 순)
- 실시간 검사 라인에서는 속도도 선택 기준이 된다

### 복습 퀴즈
1. bilateralFilter가 엣지를 보존하는 원리는? (sigmaColor의 역할)
2. medianBlur의 ksize를 짝수로 넣으면 왜 에러가 나는가?
3. 차이 이미지(absdiff)에서 결함 부분이 밝게 나오면 좋은 전처리인가, 나쁜 전처리인가?